In [25]:
import numpy as np
import pygame
import sys

# --- Running pygame windows from Jupyter (Linux, macOS, Windows) ----------------------------
# Every cell that starts with %%pygame_window runs in its own short-lived Python process.
# Closing the window (X button) simply ends that process, so the Jupyter kernel never touches
# the window system itself. Opening/closing SDL windows inside the kernel leaves frozen
# windows behind on macOS and can hang or crash the kernel.
# Definitions from previously executed cells (classes, functions, variables) are available
# in the window cell, because those cells are replayed (silently) in the new process first.
# Variables created inside a %%pygame_window cell only exist while its window is open, so put
# anything later cells need into a normal cell.
# Close a window with its X button or Esc.
import os, json, subprocess, tempfile

_PYGAME_CHILD = r'''
import contextlib, io, json, linecache, sys, traceback
with open(sys.argv[1], encoding="utf-8") as f:
    payload = json.load(f)
ns = {"__name__": "__main__"}
for src in payload["setup"]:  # replay earlier notebook cells, ignore their output and errors
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            exec(compile(src, "<notebook cell>", "exec"), ns)
    except BaseException:
        pass

# Every window opens in front with keyboard focus, and Esc closes it like the X button
# (Esc is turned into a normal pygame.QUIT event, handled by the cell's own QUIT check).
import pygame
_orig_set_mode, _orig_get = pygame.display.set_mode, pygame.event.get

def _set_mode(*args, **kwargs):
    surface = _orig_set_mode(*args, **kwargs)
    try:  # bring the new window to the front with keyboard focus
        import warnings
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            if hasattr(pygame, "Window"):  # pygame-ce
                pygame.Window.from_display_module().focus()
            else:  # classic pygame
                from pygame._sdl2.video import Window
                Window.from_display_module().focus()
    except Exception:
        pass
    return surface

def _get(*args, **kwargs):
    return [pygame.event.Event(pygame.QUIT) if e.type == pygame.KEYDOWN and e.key == pygame.K_ESCAPE else e
            for e in _orig_get(*args, **kwargs)]

pygame.display.set_mode, pygame.event.get = _set_mode, _get

main = "\n" + payload["main"]  # leading newline: line numbers match the notebook cell (line 1 = %%pygame_window)
linecache.cache["<pygame cell>"] = (len(main), None, main.splitlines(True), "<pygame cell>")
try:
    exec(compile(main, "<pygame cell>", "exec"), ns)
except SystemExit:
    raise
except BaseException as e:
    traceback.print_exception(type(e), e, e.__traceback__.tb_next)
    sys.exit(1)
'''

_OPENS_WINDOW = "display." + "set_mode("  # split so this cell does not match itself

def _replayable_cells():
    """Source of previously executed cells, without magics and without cells that open windows."""
    cells = []
    for src in get_ipython().history_manager.input_hist_raw:
        if not src.strip() or src.lstrip().startswith("%%") or _OPENS_WINDOW in src:
            continue
        lines = [l for l in src.splitlines() if not l.lstrip().startswith(("%", "!"))]
        cells.append("\n".join(lines))
    return cells

class PygameWindowError(Exception):
    pass

def pygame_window(line, cell):
    with tempfile.TemporaryDirectory() as tmp:
        payload_file = os.path.join(tmp, "payload.json")
        child_file = os.path.join(tmp, "child.py")
        with open(payload_file, "w", encoding="utf-8") as f:
            json.dump({"setup": _replayable_cells(), "main": cell}, f)
        with open(child_file, "w", encoding="utf-8") as f:
            f.write(_PYGAME_CHILD)
        env = dict(os.environ, PYGAME_HIDE_SUPPORT_PROMPT="1", PYTHONUNBUFFERED="1", PYTHONIOENCODING="utf-8")
        proc = subprocess.Popen([sys.executable, child_file, payload_file], cwd=os.getcwd(), env=env,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, encoding="utf-8", errors="replace")
        try:
            for out_line in proc.stdout:  # forward prints and tracebacks to the notebook
                print(out_line, end="")
            proc.wait()
        except KeyboardInterrupt:  # "Interrupt" button closes the window
            proc.terminate()
            proc.wait()
            print("pygame window closed by interrupt")
            return
    if proc.returncode != 0:
        raise PygameWindowError("the pygame cell raised an error, see the traceback above")

def _short_error(shell, etype, evalue, tb, tb_offset=None):
    print(f"{etype.__name__}: {evalue}", file=sys.stderr)  # the real traceback is printed above

try:
    get_ipython().register_magic_function(pygame_window, "cell", "pygame_window")
    get_ipython().set_custom_exc((PygameWindowError,), _short_error)
except NameError:
    pass  # not running inside IPython / Jupyter


def close_pygame():
    pygame.quit()

## Exercise 1: Locomotion

*A robot equipped with a differential drive starts at position $x = 1.0m$, $y = 2.0m$
and with heading $\theta = \frac{\pi}{2}$ . 
It has to move to the position $x = 1.5m$, $y = 2.0m$, $\theta = \frac{\pi}{2}$ (all angles in radians). The movement of the vehicle is described by steering commands ($v_l$ = speed of left wheel, $v_r$ = speed of right wheel, $t$ = driving time).*

- (a) *What is the minimal number of steering commands ($v_l$ , $v_r$ , $t$) needed to guide the vehicle to the desired target location?*

Only two steering commands are needed. The distance between the starting and target position is 0.5m along the x-axis. If we use this information we can set an ICC at x = 1.25 and y = 2 (halfway between the two positions). By choosing the right wheel speeds, the robot can perform a 180-degree circular motion around the ICC while maintaining a constant distance of 0.25m from it. After that the robot will end up at x = 1.5 and y = 2. At this point the robot is facing down so the second steering command is to rotate the robot by 180 degrees on the spot.

- (b) *What is the length of the shortest trajectory under this constraint?*

The lenght of the "Kreisumfang" is measured by U = 2 * $\pi$ * r. Because we do a 180 degree circular motion, we only neer U = $\pi$ * r (r = 0.25).

U = $\pi$ * 0.25

U ≈ 0.785

If the trajectory refers to the distance travelled by the centre of the axle connecting the two wheels, it is only necessary to measure U because the centre stays at the same position when the robot performs his second steering command. 
I assume that we should not calculate the distance covered by the individual wheels.

- (c) *Which sequence of steering commands guides the robot on the shortest trajectory to the desired location if an arbitrary number of steering commands can be used? The maximum velocity of each wheel is $v$ and the distance between both wheels is $l$.*

You would need 3 commands.
The first one turns the robot 90 degrees to the right ($v_l$ = v and $v_r$ = -v).

Then the robot drives to x = 1.5 and y = 2 ($v_l$ = v and $v_r$ = v).

With the last command the robot turns 90 degrees to the left to face the same direction as at the beginning ($v_l$ = -v and $v_r$ = v).

- (d) *What is the length of this trajectory?*

*Note, the length of the trajectory refers to the traveled distance along the trajectory.*

The distance that the centre of the axle moves is the difference between the starting point and the target position. 

$X_s$ = 1,  $Y_s$ = 2, $X_t$ = 1.5,  $Y_t$ = 2

$X_t$ - $X_s$ = 0.5, 
$Y_t$ - $Y_s$ = 0

The robot moves 0.5m into the x-direction and 0m into the y-direction.

## Exercise 2: Differential Drive

*Both drives in this sheet move the robot on a circle about the ICC. They only differ in how the forward velocity $v$ and the angular velocity $\omega$ follow from the commands. Write a function that computes $v$ and $\omega$ of a differential drive robot from the wheel speeds $v_l$, $v_r$ and the distance $l$ between the wheels (see lecture).*

In [44]:
def diffdrive_velocities(v_l, v_r, l):
    """Velocities of a differential drive robot.
    Args:
        v_l: speed of the left wheel in robot units per second
        v_r: speed of the right wheel in robot units per second
        l: distance between the two wheels in robot units
    Returns:
        (v, omega): forward velocity of the robot centre and angular velocity about the ICC
    """
    # forward velocity of the robot centre
    v = (v_l + v_r) / 2
    # angular velocity about the ICC
    omega = (v_r - v_l) / l
    # return v, omega (w)
    return v, omega

## Exercise 3: Ackermann Drive

*A car-like robot is driven by its rear axle with speed $v$, and its front wheels are steered by the angle $\varphi$. The pose $(x, y, \theta)$ refers to the centre of the rear axle and $d$ is the distance between front and rear axle. Write a function that computes $v$ and $\omega$ of the Ackermann drive (see lecture).*

In [45]:
def ackermann_velocities(v, phi, d):
    """Velocities of an Ackermann (car-like) robot.
    Args:
        v: forward speed of the rear axle centre in robot units per second
        phi: steering angle of the front wheels in radians
        d: distance between front and rear axle in robot units
    Returns:
        (v, omega): forward velocity of the rear axle centre and angular velocity about the ICC
    """
    # angular velocity about the ICC
    omega = (v * np.tan(phi)) / d
    # return v, omega
    return v, omega

## Exercise 4: Forward Kinematics

*Write a function that computes the new pose $\mathbf{x}_{t+1}$ from the pose $\mathbf{x}_t = (x, y, \theta)$, the velocities $v$ and $\omega$ of Exercise 2 or 3 and the driving time $\delta t$. For $\omega = 0$ the robot drives straight along its heading. Otherwise it rotates by $\omega\,\delta t$ about the ICC with $R = \frac{v}{\omega}$ and $\text{ICC} = (x - R\sin\theta,\; y + R\cos\theta)$: shift the ICC to the origin, rotate, and shift back (see lecture).*

In [46]:
def forward_kinematics(old_pose, v, omega, dt):
    """Forward kinematics: rotation of the robot about the ICC.
    Args:
        old_pose: current robot pose x_t as a Transformation instance (robot2world)
        v: forward velocity in robot units per second
        omega: angular velocity in radians per second
        dt: driving time in seconds
    Returns:
        The new robot pose x_{t+1} as a Transformation instance.
    Hint:
        Read x, y and theta from old_pose (translation and rotation.get_angle()) and treat
        omega close to 0 (straight line) separately, because the radius R becomes infinite there.
    """
    # read x, y and theta from old_pose
    x = old_pose.translation.x
    y = old_pose.translation.y
    theta = old_pose.rotation.get_angle()

    # straight line (omega close to 0, R becomes infinite)
    dtheta = omega * dt

    if np.isclose(omega, 0.0):
        x_new = x + v * dt * np.cos(theta)
        y_new = y + v * dt * np.sin(theta)
        theta_new = theta

    # circular motion about the ICC
    else:
    # radius of curvature
        R = v / omega
    # center of curvature
        icc_x = x - R * np.sin(theta)
        icc_y = y + R * np.cos(theta)

        position = np.array([x - icc_x, y - icc_y])

        rotation = Rotation.from_angle(dtheta)
        new_position = rotation.apply_to_array([Point(*position)])[0]
    # angle change
    
    # shift the ICC to the origin, rotate by dtheta, shift back
        x_new = new_position.x + icc_x
        y_new = new_position.y + icc_y
        theta_new = theta + dtheta

    # return the new pose as a Transformation
    new_pose = Transformation(
        Rotation.from_angle(theta_new),
        Point(x_new, y_new)
    )

    return new_pose

*The functions `diffdrive` and `ackermann_drive` below combine your velocity functions with the forward kinematics and are used by the simulation. Run the check cell after the `Transformation`, `Point` and `Rotation` classes to test your solution, then drive the robot in the pygame window.*

In [47]:
# Given: complete motion models used by the simulation (no changes needed)
def diffdrive(old_pose, v_l, v_r, t, l):
    v, omega = diffdrive_velocities(v_l, v_r, l)
    return forward_kinematics(old_pose, v, omega, t)

def ackermann_drive(old_pose, v, phi, dt, d):
    v, omega = ackermann_velocities(v, phi, d)
    return forward_kinematics(old_pose, v, omega, dt)

In [48]:
# Transformation, Point and Rotation classes from sheet01
class Point:
    def __init__(self, x, y):
        self.x = float(x)
        self.y = float(y)

    def __add__(self, other):
        if not isinstance(other, Point):
            return NotImplemented

        return Point(self.x + other.x, self.y + other.y)

    def __sub__(self, other):
        if not isinstance(other, Point):
            return NotImplemented

        return Point(self.x - other.x, self.y - other.y)

    def __mul__(self, second):
        if not isinstance(second, int) or isinstance(second, float):
            return NotImplemented

        return Point(self.x * second, self.y * second)
    
    def __neg__(self):
        return Point(-self.x, -self.y)

    def as_array(self):
        return np.array([self.x, self.y])

class Point_h(Point):
    def __init__(self, x, y):
        Point.__init__(self, x, y)
        self.z = 1
    def as_array(self):
        return np.array([self.x, self.y, self.z])

class Rotation:
    def __init__(self, matrix):
        matrix = np.asarray(matrix, dtype=float)

        # Check properties
        if matrix.shape != (2, 2):
            raise ValueError("A rotation matrix must have shape (2, 2).")
        if not np.allclose(matrix.T @ matrix, np.eye(2)):
            raise ValueError("Matrix must be orthogonal.")
        if not np.isclose(np.linalg.det(matrix), 1.0):
            raise ValueError("Matrix must have determinant +1.")

        self.matrix = matrix

    @classmethod
    def from_angle(cls, theta):
        c, s = np.cos(theta), np.sin(theta)
        return cls(np.array([[c, -s],
                             [s,  c]]))

    def get_angle(self):
        # In radians
        return float(np.arctan2(self.matrix[1][0], self.matrix[0][0]))

    def apply(self, point):
        if not isinstance(point, Point):
            raise TypeError("parameter must be a Point")
        point = self.matrix @ point.as_array()
        return Point(point[0], point[1])

    def apply_to_array(self, points):
        points = list(points)
        if not all(isinstance(p, Point) for p in points):
            raise TypeError("parameter must be an array of points")

        points_array = np.array([p.as_array() for p in points])

        # Points saved as row vectors (p^T) so R*p = p^T*R^T is used 
        return [Point(x, y) for x, y in points_array @ self.matrix.T]

    def inverse(self):
        return Rotation(self.matrix.T)

    def __matmul__(self, other):
        # (R1 @ R2) applies R2, then R1
        if not isinstance(other, Rotation):
            raise TypeError("parameter must be a Rotation")
        return Rotation(self.matrix @ other.matrix)

class Transformation:
    def __init__(self, rotation=None, translation=None):
        if rotation is None:
            rotation = Rotation.from_angle(0)
        if translation is None:
            translation = Point(0, 0)

        if not isinstance(rotation, Rotation):
            raise TypeError("parameter must be a Rotation")
        if not isinstance(translation, Point):
            raise TypeError("parameter must be a Point")

        matrix = np.eye(3)
        matrix[:2, :2] = rotation.matrix
        matrix[:2, 2] = translation.as_array()
        self._set_matrix(matrix)

    def _set_matrix(self, matrix):
        matrix = np.asarray(matrix, dtype=float)

        if matrix.shape != (3, 3):
            raise ValueError("A homogeneous transformation matrix must be 3×3.")
        if not np.allclose(matrix[2], [0, 0, 1]):
            raise ValueError("The last row must be [0, 0, 1].")
        if not np.allclose(matrix[:2, :2].T @ matrix[:2, :2], np.eye(2)):
            raise ValueError("The rotation part must be orthogonal.")
        if not np.isclose(np.linalg.det(matrix[:2, :2]), 1.0, atol=1e-8):
            raise ValueError("The rotation part must have determinant +1.")

        self.matrix = matrix
        self.rotation = Rotation(matrix[:2, :2])
        self.translation = Point(*matrix[:2, 2])

    @classmethod
    def from_matrix(cls, matrix):
        """Create instance from a given transformation matrix.
        Args:
            matrix: array of a (3x3) homogenous transformation matrix
        """
        obj = cls.__new__(cls)
        obj._set_matrix(matrix)
        return obj

    def apply(self, point):
        """Apply transformation to a point.
        Args:
            point: either a Point or Point_h object
        Returns: 
            The transformed Point or Point_h depending on which was input.
        """
        if isinstance(point, Point):
            rotated = self.rotation.apply(point).as_array()
            result = rotated + self.translation.as_array()
            return Point(result[0], result[1])

        elif isinstance(point, Point_h):
            result = self.matrix @ point.as_array()
            return Point_h(Point(result[0], result[1]))
        else:
            raise TypeError("parameter must be either a cartesian or a homogeneous Point")


    def compose(self, other):
        """Compose two transformation matrices to one transformation.
        Args:
            other: the second transformation which is applied first, i.e. lsr2world = rob2world.compose(lsr2rob)
        Returns:
            The composed Transformation instance
        """
        if not isinstance(other, Transformation):
            raise TypeError("parameter must be a Transformation")

        return Transformation.from_matrix(self.matrix @ other.matrix)

    def inverse(self):
        """Inverts a Transformation.
        Returns:
            The new inverted Transformation
        """
        #return Transformation.from_matrix(np.linalg.inv(self.matrix))
        return Transformation(rotation=self.rotation.inverse(), translation=-self.rotation.inverse().apply(self.translation))


In [49]:
# Check your implementation (the last test is the example from the lecture: ICC = (2,4), x_t = (2,1), omega*dt = 90 deg)
def check(name, got, expected):
    try:
        ok = np.allclose(np.asarray(got, dtype=float), np.asarray(expected, dtype=float), atol=1e-6)
    except Exception:
        ok = False
    print(("OK      " if ok else "WRONG   ") + name + (f"   got {got}, expected {expected}" if not ok else ""))

def pose_tuple(pose):
    try:
        return (pose.translation.x, pose.translation.y, pose.rotation.get_angle())
    except Exception:
        return pose

check("diffdrive_velocities straight", diffdrive_velocities(1.0, 1.0, 0.5), (1.0, 0.0))
check("diffdrive_velocities on the spot", diffdrive_velocities(-1.0, 1.0, 0.5), (0.0, 4.0))
check("ackermann_velocities straight", ackermann_velocities(2.0, 0.0, 0.5), (2.0, 0.0))
check("ackermann_velocities curve", ackermann_velocities(1.0, np.arctan(0.25), 0.5), (1.0, 0.5))

x_t = Transformation(rotation=Rotation.from_angle(0.0), translation=Point(2.0, 1.0))
check("forward_kinematics straight", pose_tuple(forward_kinematics(x_t, 1.0, 0.0, 2.0)), (4.0, 1.0, 0.0))
check("forward_kinematics example", pose_tuple(forward_kinematics(x_t, 3 * np.pi / 2, np.pi / 2, 1.0)), (5.0, 4.0, np.pi / 2))

OK      diffdrive_velocities straight
OK      diffdrive_velocities on the spot
OK      ackermann_velocities straight
OK      ackermann_velocities curve
OK      forward_kinematics straight
OK      forward_kinematics example


In [50]:
# Initial robot pose in the world.
robot2world = Transformation(rotation=Rotation.from_angle(0), translation=Point(0.0, 0.0))
noisy_robot2world = robot2world

# Robot settings
l = 0.5 # Distance between left and right wheel
d = 0.5 # Distance between front and rear axle
t = np.deg2rad(45) # Magnitude of a rotational step in radians
max_wheel_speed = 2.0 # Magnitude of a translational step in robot units
differential_drive = False # Set to False for ackermann drive

# Trajectory histories
gt_traj    = [robot2world]
noisy_traj = [robot2world]

## Collision Detection

*The robot drives inside a floor plan. Collisions are decided by the actual robot (right view): when it hits a wall, both the actual robot and the exact model of the commands (left view) stay at their old poses. Walls are the dark pixels of `map_white.png`, which is drawn centered on the world origin and covers `MAP_WIDTH` x `MAP_HEIGHT` robot units. The robot is modelled as a disk of radius `ROBOT_RADIUS`. A motion step is only carried out if the robot does not overlap a wall anywhere along the step, otherwise the robot stays at its old pose (it stops at the wall but can still turn away from it).*

In [51]:
ROBOT_RADIUS = 0.25 # Robot modelled as a disk, in robot units (half the wheel distance l)
MAP_WIDTH, MAP_HEIGHT = 13.0, 13.0 # Extent of the map image in robot units

def occupancy_from_surface(surface, threshold=128):
    """Create an occupancy grid from a pygame surface of the map.
    Args:
        surface: pygame surface of the map image (dark pixels are walls)
        threshold: gray value below which a pixel counts as a wall
    Returns:
        np.array of bools with shape (H, W), True for wall pixels, row index = image y (downwards)
    """
    rgb = pygame.surfarray.array3d(surface).astype(float) # Shape (W, H, 3)
    gray = rgb.mean(axis=2).T # Transpose to (H, W)
    return gray < threshold

def world_to_grid(x, y, occupancy, map_width=MAP_WIDTH, map_height=MAP_HEIGHT):
    """Convert world coordinates to (fractional) grid coordinates.
    Args:
        x, y: position in robot units, the map is centered on the world origin
        occupancy: occupancy grid of shape (H, W)
    Returns:
        (col, row) in pixels of the occupancy grid
    """
    H, W = occupancy.shape
    col = (x / map_width + 0.5) * W
    row = (0.5 - y / map_height) * H # Image y axis points downward
    return col, row

def is_colliding(position, occupancy, radius=ROBOT_RADIUS, map_width=MAP_WIDTH, map_height=MAP_HEIGHT):
    """Check whether a disk-shaped robot at a position overlaps a wall.
    Args:
        position: Point, robot position in world coordinates
        occupancy: occupancy grid of shape (H, W)
        radius: robot radius in robot units
    Returns:
        True if any wall pixel lies within the robot disk, outside the map is free space
    """
    H, W = occupancy.shape
    col, row = world_to_grid(position.x, position.y, occupancy, map_width, map_height)
    r_col = radius / map_width * W
    r_row = radius / map_height * H

    # Only look at the bounding box of the disk, clipped to the map
    c0, c1 = max(int(np.floor(col - r_col)), 0), min(int(np.ceil(col + r_col)) + 1, W)
    r0, r1 = max(int(np.floor(row - r_row)), 0), min(int(np.ceil(row + r_row)) + 1, H)
    if c0 >= c1 or r0 >= r1:
        return False

    rows, cols = np.mgrid[r0:r1, c0:c1]
    inside = ((cols + 0.5 - col) / r_col) ** 2 + ((rows + 0.5 - row) / r_row) ** 2 <= 1.0
    return bool(np.any(occupancy[r0:r1, c0:c1] & inside))

def move_with_collision(old_pose, new_pose, occupancy, radius=ROBOT_RADIUS):
    """Accept a motion step only if it is collision free.
    Args:
        old_pose: Transformation, current robot pose
        new_pose: Transformation, pose proposed by the motion model
        occupancy: occupancy grid of shape (H, W)
        radius: robot radius in robot units
    Returns:
        (pose, collided): new_pose and False if the step is free, otherwise old_pose and True
    """
    p0 = old_pose.translation.as_array()
    p1 = new_pose.translation.as_array()

    # Check intermediate positions so fast steps cannot tunnel through thin walls
    n_checks = max(1, int(np.ceil(np.linalg.norm(p1 - p0) / (radius / 2))))
    for s in np.linspace(0.0, 1.0, n_checks + 1)[1:]:
        p = (1 - s) * p0 + s * p1
        if is_colliding(Point(p[0], p[1]), occupancy, radius):
            return old_pose, True
    return new_pose, False

In [52]:
%%pygame_window
WIDTH, HEIGHT = 1200, 780
FPS = 20

# Color settings
BG = (24, 29, 39)
PANEL = (32, 39, 52)
GRID = (54, 64, 80)
WHITE = (235, 239, 245)
MUTED = (160, 172, 190)
BLUE = (70, 145, 255)
CYAN = (70, 220, 235)
RED = (255, 70, 85)
ORANGE = (255, 170, 65)
SOFT_ORANGE = (190, 130, 85)
GREEN = (75, 220, 140)

# Noise settings 
# Differential
WHEEL_NOISE_STD = 0.1  # Robot distance per update, random noise due to environmental factors
left_scale = np.random.normal(1.0, 0.02) # Motion model's misestimation of travel distance, persistent bias per step
right_scale = np.random.normal(1.0, 0.02)
# Ackermann
VELOCITY_NOISE_STD = 0.1
STEERING_NOISE_STD = np.deg2rad(1.0)
velocity_scale = np.random.normal(1.0, 0.02)
steering_bias = np.random.normal(0.0, np.deg2rad(0.5))

# Window resize helpers
def layout_views(width, height):
    global world_view, traj_view

    margin = 20
    gap = 20
    top = 65
    bottom_area = 150

    view_size = min((width - 2 * margin - 2 * gap) // 2, height - top - bottom_area)
    view_size = max(100, view_size)

    total_width = 2 * view_size + 2 * gap
    left = (width - total_width) // 2

    traj_view = pygame.Rect(left, top, view_size, view_size)
    world_view = pygame.Rect(left + view_size + gap, top, view_size, view_size)

def view_scale(view, world_span=12, margin=0.9):
    return min(view.width, view.height) * margin / world_span

pygame.init()
screen = pygame.display.set_mode((WIDTH, HEIGHT), pygame.RESIZABLE)
map_image = pygame.image.load("map_white.png").convert()
occupancy = occupancy_from_surface(map_image) # Walls of the map for collision detection

traj_view = pygame.Rect(20, 65, 560, 560) # Trajectory window shows robot moving according to input commands
world_view = pygame.Rect(620, 65, 560, 560) # World window shows the actual robot movement in the world
layout_views(*screen.get_size())
pygame.display.set_caption("Differential Drive Motion Model")

clock = pygame.time.Clock()
font = pygame.font.SysFont("consolas", 16)
small_font = pygame.font.SysFont("consolas", 13)
title_font = pygame.font.SysFont("consolas", 21, bold=True)

def update_robot_pose(pose):
    global robot2world, noisy_robot2world

    robot_x = pose.translation.x
    robot_y = pose.translation.y
    theta = pose.rotation.get_angle()

    robot2world = Transformation(rotation=Rotation.from_angle(theta),translation=Point(robot_x, robot_y))
    noisy_robot2world = Transformation(rotation=Rotation.from_angle(theta),translation=Point(robot_x, robot_y))

# Coordinate transformations between robot units to pygame pixels, no robot coordinate frame transformations
def world_to_screen(point):
    scale = view_scale(world_view)
    return (
        round(world_view.centerx + point.x * scale),
        round(world_view.centery - point.y * scale))# Subtract because pygame y axis points downward


def traj_to_screen(point):
    scale = view_scale(traj_view)
    return (
        round(traj_view.centerx + point.x * scale),
        round(traj_view.centery - point.y * scale),
    )

def screen_to_world(pos):
    scale = view_scale(world_view)
    return (
        (pos[0] - world_view.centerx) / scale,
        (world_view.centery - pos[1]) / scale,
    )

def screen_to_traj(pos):
    scale = view_scale(traj_view)
    return (
        (pos[0] - traj_view.centerx) / scale,
        (traj_view.centery - pos[1]) / scale,
    )

def draw_text(text, pos, color=WHITE, font_obj=font):
    surface = font_obj.render(text, True, color)
    screen.blit(surface, pos)

def draw_robot():
    origin = robot2world.apply(Point(0, 0))
    heading = robot2world.apply(Point(0.3, 0))

    p0 = traj_to_screen(origin)
    p1 = traj_to_screen(heading)
    radius = max(4, round(ROBOT_RADIUS * view_scale(traj_view))) # Drawn with the collision radius

    pygame.draw.circle(screen, BLUE, p0, radius)
    pygame.draw.circle(screen, RED if noisy_collided else WHITE, p0, radius, 2)
    pygame.draw.line(screen, ORANGE, p0, p1, 4)
    pygame.draw.circle(screen, ORANGE, p1, 4)

def draw_noisy_robot():
    origin = noisy_robot2world.apply(Point(0, 0))
    heading = noisy_robot2world.apply(Point(0.3, 0))

    p0 = world_to_screen(origin)
    p1 = world_to_screen(heading)
    radius = max(4, round(ROBOT_RADIUS * view_scale(world_view))) # Drawn with the collision radius

    pygame.draw.circle(screen, ORANGE, p0, radius)
    pygame.draw.circle(screen, RED if noisy_collided else WHITE, p0, radius, 2)
    pygame.draw.line(screen, BLUE, p0, p1, 4)
    pygame.draw.circle(screen, BLUE, p1, 4)

def draw_trajectory(trajectory, converter, color= BLUE):
    if len(trajectory) < 2:
        return

    screen_trajectory = [converter(pose.translation) for pose in trajectory]
    pygame.draw.lines(screen, color, False, screen_trajectory, 2) 

def draw_map(view):
    scale = view_scale(view)

    size = (round(MAP_WIDTH * scale), round(MAP_HEIGHT * scale))
    scaled_map = pygame.transform.smoothscale(map_image, size)

    rect = scaled_map.get_rect(center=view.center)
    screen.blit(scaled_map, rect)


# Main loop
traj_align = False
noisy_collided = False
running = True
try:
    while running:
        # Limits the loop to 20 FPS and returns elapsed seconds between updates
        dt = clock.tick(FPS) / 1000.0

        for event in pygame.event.get():

            if event.type == pygame.QUIT:
                traj_align = False
                running = False

            elif event.type == pygame.VIDEORESIZE:
                screen = pygame.display.set_mode(event.size, pygame.RESIZABLE)
                layout_views(*event.size)

            elif event.type == pygame.KEYDOWN:
                if event.key == pygame.K_t:
                    traj_align = not traj_align
                elif event.key == pygame.K_r:
                    update_robot_pose(Transformation(rotation=Rotation.from_angle(0.0),translation=Point(0, 0)))
                    gt_traj = [robot2world]
                    noisy_traj = [noisy_robot2world]

        # Active while held
        keys = pygame.key.get_pressed()

        forward = 0.0
        turn = 0.0

        if keys[pygame.K_w] or keys[pygame.K_UP]:
            forward = max_wheel_speed
        if keys[pygame.K_s] or keys[pygame.K_DOWN]:
            forward = -max_wheel_speed
        if keys[pygame.K_a] or keys[pygame.K_LEFT]:
            turn = t
        if keys[pygame.K_d] or keys[pygame.K_RIGHT]:
            turn = -t


        if differential_drive:
            v_l = forward - turn * l / 2
            v_r = forward + turn * l / 2

            # Computing noisy velocities
            # Distance
            d_l = v_l * dt
            d_r = v_r * dt

            # Gaussian noise
            d_l_noisy = left_scale * d_l + np.random.normal(0, WHEEL_NOISE_STD  * abs(d_l))
            d_r_noisy = right_scale * d_r + np.random.normal(0, WHEEL_NOISE_STD  * abs(d_r))

            v_l_noisy = d_l_noisy / dt
            v_r_noisy = d_r_noisy / dt

            # Apply differential drive motion model
            # The actual robot in the world decides about collisions, on a collision both robots stop
            noisy_robot2world, noisy_collided = move_with_collision(noisy_robot2world, diffdrive(noisy_robot2world, v_l_noisy, v_r_noisy, dt, l), occupancy)
            if not noisy_collided:
                robot2world = diffdrive(robot2world, v_l, v_r, dt, l)
            gt_traj.append(robot2world)

            noisy_traj.append(noisy_robot2world)

        # Ackermann drive
        else:            
            # Computing noisy velocities    
            v_noisy = (velocity_scale * forward + np.random.normal(0, VELOCITY_NOISE_STD * abs(forward)))
            phi_noisy = (turn + steering_bias + np.random.normal(0, STEERING_NOISE_STD))

            # Apply differential drive motion model
            # The actual robot in the world decides about collisions, on a collision both robots stop
            noisy_robot2world, noisy_collided = move_with_collision(noisy_robot2world, ackermann_drive(noisy_robot2world, v_noisy, phi_noisy, dt, d), occupancy)
            if not noisy_collided:
                robot2world = ackermann_drive(robot2world, forward, turn, dt, d)
            gt_traj.append(robot2world)

            noisy_traj.append(noisy_robot2world)


        screen.fill(BG)
        window_width, window_height = screen.get_size()

        pygame.draw.rect(screen, PANEL, traj_view, border_radius=8)
        pygame.draw.rect(screen, PANEL, world_view, border_radius=8)

        draw_text("Exact Calculation of Input Commands", (traj_view.left, 25), WHITE, title_font)
        draw_text("Actual Trajectory",(world_view.left, 25), WHITE, title_font)

        old_clip = screen.get_clip()

        # Trajectory view
        screen.set_clip(traj_view)
        draw_map(traj_view)
        draw_trajectory(gt_traj, traj_to_screen)
        if traj_align:
            draw_trajectory(noisy_traj, traj_to_screen, SOFT_ORANGE)


        draw_robot()

        # World view
        screen.set_clip(world_view)
        draw_map(world_view)
        draw_trajectory(noisy_traj, world_to_screen, ORANGE)
        draw_noisy_robot()

        screen.set_clip(old_clip)

        draw_text(f"Position: ({robot2world.translation.x:.2f}, {robot2world.translation.y:.2f}) m", (traj_view.left + 5, traj_view.bottom + 8), WHITE, small_font)
        draw_text(f"Noisy Position: ({noisy_robot2world.translation.x:.2f}, {noisy_robot2world.translation.y:.2f}) m", (world_view.left + 5, world_view.bottom + 22), WHITE, small_font)
        draw_text("Actual movement carried out by the robot in the real world", (world_view.left + 5, world_view.bottom + 8), WHITE, small_font)

        degrees = np.degrees(robot2world.rotation.get_angle())
        noisy_degrees = np.degrees(noisy_robot2world.rotation.get_angle())

        draw_text(f"Heading: {degrees:.1f} deg", (traj_view.left + 5, window_height - 100), WHITE)
        draw_text(f"Noisy Heading: {noisy_degrees:.1f} deg", (world_view.left + 5, window_height - 100), WHITE)

        if noisy_collided:
            draw_text("Collision!", (world_view.right - 100, world_view.bottom + 8), RED)

        draw_text(" Arrows and WASD: rotate and move | R: reset | T: align trajectories", (20, window_height - 30), MUTED, small_font)


        pygame.display.flip()

finally:
    close_pygame()